# 🔍 Embryoscope Silver Layer Reconciliation: Local DuckDB vs AWS Athena Production

This notebook automates the data quality reconciliation between the local **DuckDB** database (`huntington_data_lake.duckdb`, schema `silver_embryoscope`) and the production **AWS Athena** database (`silver_embryoscope_prod`).

### Objectives:
1. **Row Count Audit**: Compare total records per table.
2. **Primary Key Overlap Audit**: Reconcile exact keys present in both environments, only locally, or only in production.
3. **Unit/Location Breakdown**: Group counts by clinic server to check for isolated sync failures.
4. **Yearly Breakdown**: Drill down row counts and key matches per calendar year (using mapped date columns).
5. **Newest Record Analysis**: Fetch and show the most recent records present in only one environment (ordered by primary key descending).

### Database Connections:
- **Local**: DuckDB (`huntington_data_lake.duckdb` -> schema: `silver_embryoscope`)
- **AWS Athena**: PyAthena (`silver_embryoscope_prod` database)

In [1]:
import os
import duckdb
import pandas as pd
import numpy as np
from pyathena import connect
import warnings
warnings.filterwarnings('ignore')

# Configuration
DUCKDB_PATH = '../../database/huntington_data_lake.duckdb'
ATHENA_REGION = 'sa-east-1'
ATHENA_WORKGROUP = 'datalake-admins'
ATHENA_DB = 'silver_embryoscope_prod'

print("Libraries imported. Configured database paths:")
print(f"  Local DuckDB: {os.path.abspath(DUCKDB_PATH)}")
print(f"  AWS Athena:   {ATHENA_DB} (Region: {ATHENA_REGION}, Workgroup: {ATHENA_WORKGROUP})")

Libraries imported. Configured database paths:
  Local DuckDB: g:\My Drive\projetos_individuais\Huntington\database\huntington_data_lake.duckdb
  AWS Athena:   silver_embryoscope_prod (Region: sa-east-1, Workgroup: datalake-admins)


## 🔌 Connection Helpers
Defining wrapper functions to connect, execute queries, and guarantee connection closure.

In [2]:
def run_duck(query):
    """Runs a query on local DuckDB, ensuring connection is closed."""
    conn = duckdb.connect(DUCKDB_PATH, read_only=True)
    try:
        return conn.execute(query).df()
    finally:
        conn.close()

def run_athena(query):
    """Runs a query on AWS Athena, ensuring connection is closed."""
    conn = connect(region_name=ATHENA_REGION, work_group=ATHENA_WORKGROUP, schema_name=ATHENA_DB)
    try:
        return pd.read_sql(query, conn)
    finally:
        conn.close()

def get_common_columns(table):
    # 1. Local columns
    conn = duckdb.connect(DUCKDB_PATH, read_only=True)
    try:
        local_cols = set([c[0].lower() for c in conn.execute(f"SELECT * FROM silver_embryoscope.{table} LIMIT 0").description])
    finally:
        conn.close()
    
    # 2. Athena columns
    try:
        prod_df = run_athena(f"SELECT * FROM silver_embryoscope_prod.{table} LIMIT 0")
        prod_cols = set([c.lower() for c in prod_df.columns])
    except Exception:
        prod_cols = set()
        
    return list(local_cols & prod_cols)

# Test connections
try:
    duck_ok = run_duck("SELECT 1 as test").iloc[0]['test'] == 1
    print("✅ DuckDB Connection: OK")
except Exception as e:
    print(f"❌ DuckDB Connection: FAILED - {e}")
    duck_ok = False

try:
    athena_ok = run_athena("SELECT 1 as test").iloc[0]['test'] == 1
    print("✅ AWS Athena Connection: OK")
except Exception as e:
    print(f"❌ AWS Athena Connection: FAILED - {e}")
    athena_ok = False

✅ DuckDB Connection: OK
✅ AWS Athena Connection: OK


## 🗺️ Configuration & Predefined Mappings
Defining primary keys, date columns, and location/server fields.

In [3]:
TABLES_CONFIG = {
    "patients": {
        "local_table": "patients",
        "athena_table": "patients",
        "keys": ["PatientIDx"],
        "target_keys": ["patient_id_x"],
        "date_col": "DateOfBirth",
        "target_date_col": "date_of_birth",
        "local_loc_col": "_location",
        "target_loc_col": "source_server"
    },
    "treatments": {
        "local_table": "treatments",
        "athena_table": "treatments",
        "keys": ["PatientIDx", "TreatmentName"],
        "target_keys": ["patient_id_x", "treatment_name"],
        "date_col": "_extraction_timestamp",
        "target_date_col": "bronze_updated_at",
        "local_loc_col": "unit_huntington",
        "target_loc_col": "source_server"
    },
    "embryo_data": {
        "local_table": "embryo_data",
        "athena_table": "embryo_data",
        "keys": ["EmbryoID"],
        "target_keys": ["embryo_id"],
        "date_col": "KIDDate",
        "target_date_col": "kid_date",
        "local_loc_col": "unit_huntington",
        "target_loc_col": "unit_huntington"
    }
}
print("Configured key, date, and location mappings for Embryoscope tables.")

Configured key, date, and location mappings for Embryoscope tables.


## 📊 Part 1: Row Count & Key Reconciliation Summary
Iterating through all configured tables to compare total rows, match counts, and overlaps.

In [4]:
def normalize_key(val):
    if val is None or pd.isna(val):
        return None
    val_str = str(val).strip().lower()
    if val_str.endswith('.0'):
        val_str = val_str[:-2]
    if val_str.isdigit():
        try:
            val_str = str(int(val_str))
        except ValueError:
            pass
    if " 00:00:00" in val_str:
        val_str = val_str.replace(" 00:00:00", "")
    return val_str

summary_data = []
for t, cfg in TABLES_CONFIG.items():
    local_tbl = f"silver_embryoscope.{cfg['local_table']}"
    target_tbl = f"silver_embryoscope_prod.{cfg['athena_table']}"
    l_keys = cfg['keys']
    t_keys = cfg['target_keys']
    
    l_select_sql = ", ".join([f'"{c}"' for c in l_keys])
    t_select_sql = ", ".join([f'"{c}"' for c in t_keys])
    
    local_count = run_duck(f"SELECT COUNT(*) as cnt FROM {local_tbl}").iloc[0]['cnt']
    prod_count = run_athena(f"SELECT COUNT(*) as cnt FROM {target_tbl}").iloc[0]['cnt']
    
    local_df = run_duck(f"SELECT {l_select_sql} FROM {local_tbl}")
    local_df.columns = [c.lower() for c in local_df.columns]
    
    prod_df = run_athena(f"SELECT {t_select_sql} FROM {target_tbl}")
    prod_df.columns = [c.lower() for c in prod_df.columns]
    
    l_keys_lower = [k.lower() for k in l_keys]
    t_keys_lower = [tk.lower() for tk in t_keys]
    
    local_df['_comp_key'] = local_df.apply(
        lambda r: "||".join([normalize_key(r[k]) if normalize_key(r[k]) is not None else "NULL" for k in l_keys_lower]), axis=1
    )
    prod_df['_comp_key'] = prod_df.apply(
        lambda r: "||".join([normalize_key(r[k]) if normalize_key(r[k]) is not None else "NULL" for k in t_keys_lower]), axis=1
    )
    
    local_keys_set = set(local_df['_comp_key'].dropna().tolist())
    prod_keys_set = set(prod_df['_comp_key'].dropna().tolist())
    
    matched_keys = len(local_keys_set & prod_keys_set)
    only_local = len(local_keys_set - prod_keys_set)
    only_prod = len(prod_keys_set - local_keys_set)
    
    summary_data.append({
        'Table': t,
        'Local Rows': local_count,
        'Athena Rows': prod_count,
        'Difference': local_count - prod_count,
        'Match Count': matched_keys,
        'Only in Local (DuckDB)': only_local,
        'Only in Prod (Athena)': only_prod
    })

summary_df = pd.DataFrame(summary_data)
summary_df.style.format({
    'Local Rows': '{:,}',
    'Athena Rows': '{:,}',
    'Difference': '{:+,}',
    'Match Count': '{:,}',
    'Only in Local (DuckDB)': '{:,}',
    'Only in Prod (Athena)': '{:,}'
}).bar(subset=['Difference'], align='mid', color=['#d65f5f', '#5fba7d'])

,Table,Local Rows,Athena Rows,Difference,Match Count,Only in Local (DuckDB),Only in Prod (Athena)
0,patients,"14,179","14,175",+4,"14,175",4,0
1,treatments,"23,856","23,768",+88,"23,768",88,0
2,embryo_data,"152,327","152,293",+34,"152,277",50,16


## 🏥 Part 1.2: Location/Unit Breakdown
Grouping records by location server to pinpoint which clinics are out of sync.

In [5]:
for t, cfg in TABLES_CONFIG.items():
    local_tbl = f"silver_embryoscope.{cfg['local_table']}"
    target_tbl = f"silver_embryoscope_prod.{cfg['athena_table']}"
    loc_col = cfg['local_loc_col']
    t_loc_col = cfg['target_loc_col']
    
    print("=" * 80)
    print(f"🏥 Table: {t} | Unit Breakdown ({loc_col} vs {t_loc_col})")
    print("=" * 80)
    
    duck_loc = run_duck(f"SELECT \"{loc_col}\" as unit, COUNT(*) as local_cnt FROM {local_tbl} GROUP BY 1")
    ath_loc = run_athena(f"SELECT \"{t_loc_col}\" as unit, COUNT(*) as athena_cnt FROM {target_tbl} GROUP BY 1")
    
    merged_loc = pd.merge(duck_loc, ath_loc, on='unit', how='outer').fillna(0)
    merged_loc['local_cnt'] = merged_loc['local_cnt'].astype(int)
    merged_loc['athena_cnt'] = merged_loc['athena_cnt'].astype(int)
    merged_loc['diff'] = merged_loc['local_cnt'] - merged_loc['athena_cnt']
    
    display(merged_loc)
    print("\n")

🏥 Table: patients | Unit Breakdown (_location vs source_server)


,unit,local_cnt,athena_cnt,diff
0,Belo Horizonte,2590,2590,0
1,Brasilia,1807,1807,0
2,Ibirapuera,6180,6176,4
3,Salvador,242,242,0
4,Vila Mariana,3360,3360,0




🏥 Table: treatments | Unit Breakdown (unit_huntington vs source_server)


,unit,local_cnt,athena_cnt,diff
0,Belo Horizonte,3431,3431,0
1,Brasilia,2460,2460,0
2,Ibirapuera,13212,13124,88
3,Salvador,280,280,0
4,Vila Mariana,4473,4473,0




🏥 Table: embryo_data | Unit Breakdown (unit_huntington vs unit_huntington)


,unit,local_cnt,athena_cnt,diff
0,Belo Horizonte,27213,27213,0
1,Brasilia,18446,18446,0
2,Ibirapuera,66493,66459,34
3,Salvador,2274,2274,0
4,Vila Mariana,37901,37901,0


## 📅 Part 2: Yearly Breakdown Analysis
Drilling down row counts and key overlaps per year for each table using date/timestamp columns.

In [6]:
# Part 2: Year Breakdown — embryo_data only (year stripped from embryo_id)
t = "embryo_data"
cfg = TABLES_CONFIG[t]

local_tbl = f"silver_embryoscope.{cfg['local_table']}"
target_tbl = f"silver_embryoscope_prod.{cfg['athena_table']}"

l_key = cfg['keys'][0]        # EmbryoID
t_key = cfg['target_keys'][0] # embryo_id
loc_col = cfg['local_loc_col']   # unit_huntington
t_loc_col = cfg['target_loc_col']  # unit_huntington

print("=" * 80)
print(f"📅 Table: {t} — Year Breakdown (year stripped from embryo_id)")
print("=" * 80)

# Fetch local data
local_df = run_duck(f'SELECT "{l_key}", "{loc_col}" FROM {local_tbl}')
local_df.columns = [c.lower() for c in local_df.columns]
local_df['_norm_key'] = local_df[l_key.lower()].apply(normalize_key)

# Fetch Athena data
prod_df = run_athena(f'SELECT "{t_key}", "{t_loc_col}" FROM {target_tbl}')
prod_df.columns = [c.lower() for c in prod_df.columns]
prod_df['_norm_key'] = prod_df[t_key.lower()].apply(normalize_key)

# Strip year from embryo_id: take the first 4 characters
# EmbryoID format is expected to start with the year, e.g. "2023-..."
def extract_year_from_id(eid):
    # EmbryoID format: DYYYY.MM.DD_... (e.g. D2020.05.13_S00040_I4120_P-1)
    if eid is None or (isinstance(eid, float) and pd.isna(eid)):
        return "N/A"
    s = str(eid).strip()
    if len(s) >= 5 and s[0].upper() == 'D' and s[1:5].isdigit():
        return s[1:5]
    return "N/A"

def extract_yearmonth_from_id(eid):
    # EmbryoID format: DYYYY.MM.DD_... -> returns "YYYY-MM"
    if eid is None or (isinstance(eid, float) and pd.isna(eid)):
        return "N/A"
    s = str(eid).strip()
    # Expected: D2020.05.13_...
    if len(s) >= 8 and s[0].upper() == 'D' and s[1:5].isdigit() and s[5] == '.' and s[6:8].isdigit():
        return f"{s[1:5]}-{s[6:8]}"
    return "N/A"

local_df['year'] = local_df[l_key.lower()].apply(extract_year_from_id)
prod_df['year'] = prod_df[t_key.lower()].apply(extract_year_from_id)

years = sorted(set(local_df['year'].tolist()) | set(prod_df['year'].tolist()))
units = sorted(set(local_df[loc_col.lower()].dropna().tolist()) | set(prod_df[t_loc_col.lower()].dropna().tolist()))

# --- Pivot: Local Count by Year x Unit ---
print("\n### Local DuckDB — Row Count by Year × Unit\n")
local_pivot_rows = []
for yr in years:
    row = {'Year': yr}
    yr_df = local_df[local_df['year'] == yr]
    for u in units:
        row[u] = int((yr_df[loc_col.lower()] == u).sum())
    row['Total'] = int(len(yr_df))
    local_pivot_rows.append(row)
local_pivot = pd.DataFrame(local_pivot_rows).set_index('Year')
display(local_pivot)

# --- Pivot: Athena Count by Year x Unit ---
print("\n### Athena Production — Row Count by Year × Unit\n")
prod_pivot_rows = []
for yr in years:
    row = {'Year': yr}
    yr_df = prod_df[prod_df['year'] == yr]
    for u in units:
        row[u] = int((yr_df[t_loc_col.lower()] == u).sum())
    row['Total'] = int(len(yr_df))
    prod_pivot_rows.append(row)
prod_pivot = pd.DataFrame(prod_pivot_rows).set_index('Year')
display(prod_pivot)

# --- Delta Pivot: Local - Athena ---
print("\n### Delta (Local − Athena) by Year × Unit\n")
delta_pivot = local_pivot.subtract(prod_pivot, fill_value=0).astype(int)
display(delta_pivot)


📅 Table: embryo_data — Year Breakdown (year stripped from embryo_id)

### Local DuckDB — Row Count by Year × Unit



,Belo Horizonte,Brasilia,Ibirapuera,Salvador,Vila Mariana,Total
Year,,,,,,
2017,0,0,602,0,0,602
2018,0,0,4127,0,0,4127
2019,2079,0,10013,0,2367,14459
2020,2946,3134,7753,0,3515,17348
2021,3731,3997,8824,0,4580,21132
2022,3528,4420,7118,0,4646,19712
2023,3825,1893,7667,0,5830,19215
2024,3991,2093,5924,0,5274,17282
2025,4682,1772,8538,1424,7154,23570



### Athena Production — Row Count by Year × Unit



,Belo Horizonte,Brasilia,Ibirapuera,Salvador,Vila Mariana,Total
Year,,,,,,
2017,0,0,602,0,0,602
2018,0,0,4127,0,0,4127
2019,2079,0,10013,0,2367,14459
2020,2946,3134,7753,0,3515,17348
2021,3731,3997,8824,0,4580,21132
2022,3528,4420,7118,0,4646,19712
2023,3825,1893,7667,0,5830,19215
2024,3991,2093,5924,0,5274,17282
2025,4682,1772,8495,1424,7154,23527



### Delta (Local − Athena) by Year × Unit



,Belo Horizonte,Brasilia,Ibirapuera,Salvador,Vila Mariana,Total
Year,,,,,,
2017,0,0,0,0,0,0
2018,0,0,0,0,0,0
2019,0,0,0,0,0,0
2020,0,0,0,0,0,0
2021,0,0,0,0,0,0
2022,0,0,0,0,0,0
2023,0,0,0,0,0,0
2024,0,0,0,0,0,0
2025,0,0,43,0,0,43


## 🔬 Part 2.2: Zoom-In Ibirapuera Yearly Breakdown
Reconciling years exclusively for the **Ibirapuera** unit where the discrepancies are concentrated.

In [7]:
# Part 2.2: Zoom-In Ibirapuera — Year-Month Breakdown (from embryo_id DYYYY.MM.DD_...)
t = "embryo_data"
cfg = TABLES_CONFIG[t]

local_tbl = f"silver_embryoscope.{cfg['local_table']}"
target_tbl = f"silver_embryoscope_prod.{cfg['athena_table']}"
loc_col = cfg['local_loc_col']    # unit_huntington
t_loc_col = cfg['target_loc_col'] # unit_huntington
l_key = cfg['keys'][0]            # EmbryoID
t_key = cfg['target_keys'][0]     # embryo_id

print("=" * 80)
print(f"\U0001f50d Table: {t} | Ibirapuera Only — YYYY-MM Breakdown from embryo_id")
print("=" * 80)

local_df = run_duck(
    f'SELECT "{l_key}", "{loc_col}" FROM {local_tbl} WHERE "{loc_col}" = \'Ibirapuera\''
)
local_df.columns = [c.lower() for c in local_df.columns]
local_df['_comp_key'] = local_df[l_key.lower()].apply(normalize_key)

prod_df = run_athena(
    f'SELECT "{t_key}", "{t_loc_col}" FROM {target_tbl} WHERE "{t_loc_col}" = \'Ibirapuera\''
)
prod_df.columns = [c.lower() for c in prod_df.columns]
prod_df['_comp_key'] = prod_df[t_key.lower()].apply(normalize_key)

local_df['year_month'] = local_df[l_key.lower()].apply(extract_yearmonth_from_id)
prod_df['year_month'] = prod_df[t_key.lower()].apply(extract_yearmonth_from_id)

year_months = sorted(set(local_df['year_month'].tolist()) | set(prod_df['year_month'].tolist()))

yearly_summary = []
for ym in year_months:
    l_keys_set = set(local_df[local_df['year_month'] == ym]['_comp_key'].dropna().tolist())
    p_keys_set = set(prod_df[prod_df['year_month'] == ym]['_comp_key'].dropna().tolist())

    matched = len(l_keys_set & p_keys_set)
    only_l = len(l_keys_set - p_keys_set)
    only_p = len(p_keys_set - l_keys_set)

    yearly_summary.append({
        'Year-Month': ym,
        'Local Count': len(l_keys_set),
        'Athena Count': len(p_keys_set),
        'Matched Count': matched,
        'Only Local': only_l,
        'Only Athena': only_p
    })

df_ys = pd.DataFrame(yearly_summary)
display(df_ys[(df_ys['Only Local']>0)|(df_ys['Only Athena']>0)])


🔍 Table: embryo_data | Ibirapuera Only — YYYY-MM Breakdown from embryo_id


,Year-Month,Local Count,Athena Count,Matched Count,Only Local,Only Athena
87,2025-02,631,616,616,15,0
88,2025-03,610,601,601,9,0
90,2025-05,749,742,742,7,0
94,2025-09,802,794,794,8,0
95,2025-10,744,740,740,4,0
104,2026-07,773,766,766,7,0
105,2026-08,174,190,174,0,16


## 🆕 Part 3: Mismatch Drill-Down — Newest Mismatched Records
Displaying up to 5 newest records unique to DuckDB or Athena for debugging purposes.

In [8]:
for t, cfg in TABLES_CONFIG.items():
    local_tbl = f"silver_embryoscope.{cfg['local_table']}"
    target_tbl = f"silver_embryoscope_prod.{cfg['athena_table']}"
    l_keys = cfg['keys']
    t_keys = cfg['target_keys']
    key = l_keys[0]
    t_key = t_keys[0]
    
    l_keys_lower = [k.lower() for k in l_keys]
    t_keys_lower = [tk.lower() for tk in t_keys]
    
    local_df = run_duck(f"SELECT \"{key}\" FROM {local_tbl}")
    local_df.columns = [c.lower() for c in local_df.columns]
    local_keys_set = set(local_df[key.lower()].dropna().tolist())
    
    prod_df = run_athena(f"SELECT \"{t_key}\" FROM {target_tbl}")
    prod_df.columns = [c.lower() for c in prod_df.columns]
    prod_keys_set = set(prod_df[t_key.lower()].dropna().tolist())
    
    only_l = sorted(list(local_keys_set - prod_keys_set), reverse=True)
    only_p = sorted(list(prod_keys_set - local_keys_set), reverse=True)
    
    print("=" * 80)
    print(f"🔍 Mismatch Samples: {t} (Primary Key: {key})")
    print("=" * 80)
    
    if only_l:
        sample_keys = only_l[:50]
        keys_ph = ", ".join([str(k) if isinstance(k, (int, float)) else f"'{k}'" for k in sample_keys])
        local_samples = run_duck(f"SELECT * FROM {local_tbl} WHERE \"{key}\" IN ({keys_ph}) ORDER BY \"{key}\" DESC")
        print(f"🆕 Top {len(sample_keys)} NEWEST records ONLY found in Local DuckDB (Total: {len(only_l)}):")
        display(local_samples)
    else:
        print("✅ No records found exclusively in Local DuckDB.")
        
    if only_p:
        sample_keys = only_p[:50]
        keys_ph = ", ".join([str(k) if isinstance(k, (int, float)) else f"'{k}'" for k in sample_keys])
        prod_samples = run_athena(f"SELECT * FROM {target_tbl} WHERE \"{t_key}\" IN ({keys_ph}) ORDER BY \"{t_key}\" DESC")
        print(f"🆕 Top {len(sample_keys)} NEWEST records ONLY found in Athena Production (Total: {len(only_p)}):")
        display(prod_samples)
    else:
        print("✅ No records found exclusively in Athena Production.")
    print("\n")

🔍 Mismatch Samples: patients (Primary Key: PatientIDx)
🆕 Top 4 NEWEST records ONLY found in Local DuckDB (Total: 4):


,PatientIDx,PatientID,FirstName,LastName,_extraction_timestamp,_location,_run_id,_row_hash,DateOfBirth,prontuario,unit_huntington
0,PC1P7BHG_45702.3600089815,865375,"OLIVEIRA, MARIANA C. L. M.",18/08/1980,2025-07-15 19:21:41.464134,Ibirapuera,8c87efb7-8ede-4df5-9401-850c4ff74b87,c08ef7b9e7cb784e28446526eb5b5f16,1980-08-18,865375,Ibirapuera
1,PC1P7BHG_45701.5396938542,835921,"GUEDES, FLAVIA da S. C.",06/01/1989,2025-07-15 19:21:41.464134,Ibirapuera,8c87efb7-8ede-4df5-9401-850c4ff74b87,a46ac6d2fd5cdca1c5e87f6688293d41,1989-01-06,835921,Ibirapuera
2,PC1P7BHG_45701.4950559607,868185,"VAICHERT, DANIELA C. de O.",07/05/1981,2025-07-15 19:21:41.464134,Ibirapuera,8c87efb7-8ede-4df5-9401-850c4ff74b87,ea86d3a46fd889e50f41c94c821382d0,1981-05-07,868185,Ibirapuera
3,PC1P7BHG_45700.5244824074,844973,"BRESSANIN, YAMILET L. B.",18/07/1979,2025-07-15 19:21:41.464134,Ibirapuera,8c87efb7-8ede-4df5-9401-850c4ff74b87,0ea29d97de4de2621529c3ed85ba7ff9,1979-07-18,844973,Ibirapuera


✅ No records found exclusively in Athena Production.


🔍 Mismatch Samples: treatments (Primary Key: PatientIDx)
🆕 Top 4 NEWEST records ONLY found in Local DuckDB (Total: 4):


,PatientIDx,TreatmentName,_extraction_timestamp,_location,_run_id,_row_hash,unit_huntington
0,PC1P7BHG_45702.3600089815,2025-1077,2025-07-15 19:21:41.464134,Ibirapuera,8c87efb7-8ede-4df5-9401-850c4ff74b87,83d0f24a29b8f2d6a2a57a34e11efd7c,Ibirapuera
1,PC1P7BHG_45702.3600089815,2025-263,2025-07-15 19:21:41.464134,Ibirapuera,8c87efb7-8ede-4df5-9401-850c4ff74b87,53654733ac4278ddaf969e6b4a2904f4,Ibirapuera
2,PC1P7BHG_45701.5396938542,2025-1043,2025-07-15 19:21:41.464134,Ibirapuera,8c87efb7-8ede-4df5-9401-850c4ff74b87,c7da921d5ffdee9fca9e6c88dba98d85,Ibirapuera
3,PC1P7BHG_45701.5396938542,2025-2211,2025-10-14 19:22:50.061926,Ibirapuera,e30a4403-235b-45bd-a7f1-53830fd4890d,375207b09f6cd08ba3aaff8b31eb2b03,Ibirapuera
4,PC1P7BHG_45701.5396938542,2025-253,2025-07-15 19:21:41.464134,Ibirapuera,8c87efb7-8ede-4df5-9401-850c4ff74b87,e553bae2423ea8e93c3c948984040b2a,Ibirapuera
5,PC1P7BHG_45701.4950559607,2025-252,2025-07-15 19:21:41.464134,Ibirapuera,8c87efb7-8ede-4df5-9401-850c4ff74b87,8a770e992c47630305eeff0373cd0598,Ibirapuera
6,PC1P7BHG_45701.4950559607,2025-527,2025-07-15 19:21:41.464134,Ibirapuera,8c87efb7-8ede-4df5-9401-850c4ff74b87,9d5effd651776f99c14a7ab22d4ba10b,Ibirapuera
7,PC1P7BHG_45700.5244824074,2025 - 2081,2025-09-19 19:28:09.359556,Ibirapuera,f6fec973-b22e-4b73-a4b3-813f1453d75f,4b08d8ebf9eda328ca6fd66596da4ec4,Ibirapuera
8,PC1P7BHG_45700.5244824074,2025-246,2025-07-15 19:21:41.464134,Ibirapuera,8c87efb7-8ede-4df5-9401-850c4ff74b87,2db9e1db76f7568f231e96f2b53e3991,Ibirapuera


✅ No records found exclusively in Athena Production.


🔍 Mismatch Samples: embryo_data (Primary Key: EmbryoID)
🆕 Top 50 NEWEST records ONLY found in Local DuckDB (Total: 50):


,EmbryoID,PatientIDx,TreatmentName,KIDDate,KIDScore,KIDUser,KIDVersion,Description,EmbryoDescriptionID,EmbryoFate,...,Time_BlastomereSize,Time_Fragmentation,Time_MultiNucleation,Timestamp_BlastomereSize,Timestamp_Fragmentation,Timestamp_MultiNucleation,Value_BlastomereSize,Value_Fragmentation,Value_MultiNucleation,embryo_number
0,D2026.07.29_S05090_I3166_P-7,PC1P7BHG_46155.4248835764,2026-1007,NaT,None,None,None,None,AB7,Unknown,...,NaN,NaN,NaN,None,None,None,None,None,None,11
1,D2026.07.29_S05089_I3166_P-5,PC1P7BHG_46108.4636951968,2026-621,NaT,None,None,None,None,AC5,Unknown,...,NaN,NaN,NaN,None,None,None,None,None,None,21
2,D2026.07.27_S04803_I3027_P-7,PC1P7BHG_46104.5466231597,2026-1033,NaT,None,None,None,None,AC7,Unknown,...,NaN,NaN,NaN,None,None,None,None,None,None,10
3,D2026.07.27_S04800_I3027_P-10,PC1P7BHG_46076.5156899653,2026-877,NaT,None,None,None,None,AC10,FrozenEmbryoTransfer,...,NaN,NaN,NaN,None,None,None,None,None,None,16
4,D2026.07.27_S04799_I3027_P-10,PC1P7BHG_46168.4420869792,1113-2026,NaT,None,None,None,None,AB10,Unknown,...,NaN,NaN,NaN,None,None,None,None,None,None,11
5,D2026.07.27_S04798_I3027_P-7,PC1P7BHG_46056.4528425926,2026-1288,NaT,None,None,None,None,AC7,FrozenEmbryoTransfer,...,NaN,NaN,NaN,None,None,None,None,None,None,16
6,D2026.07.25_S05076_I3166_P-7,PC1P7BHG_46198.4969735185,2026-1285,NaT,None,None,None,None,AC7,Transfer,...,NaN,NaN,NaN,None,None,None,None,None,None,19
7,D2025.10.02_S04351_I3027_P-4,PC1P7BHG_45701.5396938542,2025-2211,2025-10-08,5.8,ADMIN,KIDScoreD5 v3.3,Sem analise,AC4,Freeze,...,NaN,NaN,NaN,None,None,None,None,None,None,4
8,D2025.10.02_S04351_I3027_P-3,PC1P7BHG_45701.5396938542,2025-2211,2025-10-08,1.6,ADMIN,KIDScoreD5 v3.3,None,AC3,Avoid,...,NaN,NaN,NaN,None,None,None,None,None,None,3
9,D2025.10.02_S04351_I3027_P-2,PC1P7BHG_45701.5396938542,2025-2211,2025-10-08,6.7,ADMIN,KIDScoreD5 v3.3,"Euploid, XY",AC2,Freeze,...,NaN,NaN,NaN,None,None,None,None,None,None,2


🆕 Top 16 NEWEST records ONLY found in Athena Production (Total: 16):


,embryo_data_sk,embryo_id,patient_id_x,prontuario,treatment_name,embryo_number,kid_date,kid_score,kid_user,kid_version,...,time_multi_nucleation,timestamp_blastomere_size,timestamp_fragmentation,timestamp_multi_nucleation,value_blastomere_size,value_fragmentation,value_multi_nucleation,bronze_updated_at,_dlt_id,is_recovery_record
0,Ibirapuera|D2026.08.06_S05114_I3166_P-9,D2026.08.06_S05114_I3166_P-9,PC1P7BHG_46240.6293482407,973259,2026-1465,9,None,None,None,None,...,None,None,None,None,None,None,None,2026-08-07 03:08:33.286,f3oQ74tbNZDPcA,False
1,Ibirapuera|D2026.08.06_S05114_I3166_P-8,D2026.08.06_S05114_I3166_P-8,PC1P7BHG_46240.6293482407,973259,2026-1465,8,None,None,None,None,...,None,None,None,None,None,None,None,2026-08-07 03:08:33.286,nQU1ttBEzZ3sNA,False
2,Ibirapuera|D2026.08.06_S05114_I3166_P-7,D2026.08.06_S05114_I3166_P-7,PC1P7BHG_46240.6293482407,973259,2026-1465,7,None,None,None,None,...,None,None,None,None,None,None,None,2026-08-07 03:08:33.286,KrBjZBKPmbm3AQ,False
3,Ibirapuera|D2026.08.06_S05114_I3166_P-6,D2026.08.06_S05114_I3166_P-6,PC1P7BHG_46240.6293482407,973259,2026-1465,6,None,None,None,None,...,None,None,None,None,None,None,None,2026-08-07 03:08:33.286,vibs6bR8Q52J/Q,False
4,Ibirapuera|D2026.08.06_S05114_I3166_P-5,D2026.08.06_S05114_I3166_P-5,PC1P7BHG_46240.6293482407,973259,2026-1465,5,None,None,None,None,...,None,None,None,None,None,None,None,2026-08-07 03:08:33.286,rClw4iErkXTrrQ,False
5,Ibirapuera|D2026.08.06_S05114_I3166_P-4,D2026.08.06_S05114_I3166_P-4,PC1P7BHG_46240.6293482407,973259,2026-1465,4,None,None,None,None,...,None,None,None,None,None,None,None,2026-08-07 03:08:33.286,rEARh48i6sVzJA,False
6,Ibirapuera|D2026.08.06_S05114_I3166_P-3,D2026.08.06_S05114_I3166_P-3,PC1P7BHG_46240.6293482407,973259,2026-1465,3,None,None,None,None,...,None,None,None,None,None,None,None,2026-08-07 03:08:33.286,yyJK4rNUqXNs/g,False
7,Ibirapuera|D2026.08.06_S05114_I3166_P-2,D2026.08.06_S05114_I3166_P-2,PC1P7BHG_46240.6293482407,973259,2026-1465,2,None,None,None,None,...,None,None,None,None,None,None,None,2026-08-07 03:08:33.286,NMf836jX7cwAfQ,False
8,Ibirapuera|D2026.08.06_S05114_I3166_P-16,D2026.08.06_S05114_I3166_P-16,PC1P7BHG_46240.6293482407,973259,2026-1465,16,None,None,None,None,...,None,None,None,None,None,None,None,2026-08-07 03:08:33.286,i9LEyQuJS0Pujg,False
9,Ibirapuera|D2026.08.06_S05114_I3166_P-15,D2026.08.06_S05114_I3166_P-15,PC1P7BHG_46240.6293482407,973259,2026-1465,15,None,None,None,None,...,None,None,None,None,None,None,None,2026-08-07 03:08:33.286,YFsBHV3vyJHuJQ,False


In [9]:
local_samples[['EmbryoID','PatientIDx','TreatmentName']].head(6)

,EmbryoID,PatientIDx,TreatmentName
0,D2026.07.29_S05090_I3166_P-7,PC1P7BHG_46155.4248835764,2026-1007
1,D2026.07.29_S05089_I3166_P-5,PC1P7BHG_46108.4636951968,2026-621
2,D2026.07.27_S04803_I3027_P-7,PC1P7BHG_46104.5466231597,2026-1033
3,D2026.07.27_S04800_I3027_P-10,PC1P7BHG_46076.5156899653,2026-877
4,D2026.07.27_S04799_I3027_P-10,PC1P7BHG_46168.4420869792,1113-2026
5,D2026.07.27_S04798_I3027_P-7,PC1P7BHG_46056.4528425926,2026-1288
